# RSNA Knee Abnormality Detection: standalone maximum-accuracy notebook (`PRESET = "ultra"`)

**Standalone:** one file, no repository, no pip installs, no internet needed. The multilingual report labeler is embedded; the competition data,
a Qwen2.5-Instruct model and the pretrained backbone weights come from `/kaggle/input` (see section 0 for exactly what to attach and
how to create the backbone-weights dataset once). A pre-flight cell checks all of it before any slow step.

This is the same notebook as `rsna_knee_kaggle_v2` with every setting pushed toward accuracy and away from speed:
384 px, 96 slices, 48 windows per study, a CoAtNet (the Raptor backbone) plus a ConvNeXt-small arm fused on
out-of-fold predictions, 8 epochs, EMA 0.999, a 3-layer transformer, mirror TTA, and the LLM labeler switched on.

## About the 99 % target
**99 % macro accuracy is a target, not something this notebook can promise, and the notebook will not claim it.** What it
does is print a `TARGET CHECK` line with the measured number next to the references that bound it:
* the **always-negative floor** on the 58 expert studies (about 0.65-0.71), and
* the **accuracy of the training labels themselves** against the expert labels: the model is taught from report-derived
  labels, so this number is an upper-bound indicator. In the author's study the rule labels reach only 0.735 AUC against
  the experts, and the best public notebooks (about 0.95 AUC on the leaderboard) used LLM labels plus extra data.
* With 58 expert studies one error in one label moves accuracy by 0.14 %, and the AUC 95 % CI is about ±0.06, so
  "99 %" cannot even be *verified* on that set with statistical confidence.

## Two-run protocol (recommended)
1. **Run A (evaluation):** `INCLUDE_GOLD = False`. Trains on report labels only and reports honest expert-set metrics.
2. **Run B (submission):** `INCLUDE_GOLD = True`, `RESUME_DIR = None`. Adds the 58 expert studies to training; no expert-set
   metric is valid afterwards. Use Run A's numbers to describe Run B's expected quality.

## Time
The `ultra` preset is heavy. On a Kaggle T4x2 only a few folds will fit in the 8.6 h budget (an estimate, not a
measurement). Training is **fold-major** so every finished fold has both arms, per-fold checkpoints and OOF state are
saved, and a later session can continue: attach this run's output as a dataset and set `RESUME_DIR` to its path.
A faster GPU (A100/H100) lets all five folds finish.

# RSNA Knee Abnormality Detection v2: slot-based 2.5-D MIL with per-finding attention

Self-contained Kaggle notebook: turn on a **GPU** (T4×2 or P100) and **Run All**.

## What changed against v1 (and where each idea comes from)

| Limit in v1 | v2 | Source of the idea |
|---|---|---|
| 1 series per plane, 16 slices, 224 px | **5 slots** (sag fluid, sag non-fluid, cor fluid, cor non-fluid, axial), 96 slices, 256–384 px, windows of 3 adjacent slices | Raptor CoAtNet notebooks |
| Resize by pixel count | **Physical crop** (`CROP_MM` mm around the centre, from `PixelSpacing`), series ordered along the slice normal, 2–98 % span, per-series percentile window | Raptor / ConvNeXt-MIL |
| Mean over slices per plane | **Per-finding attention over all windows** + slot and slice-position embeddings + a 2-layer transformer, plus v1's global branch and deep supervision | Raptor, ConvNeXt-MIL, v1 |
| One small backbone (EfficientNet-B0) | any timm backbone; **several arms** can be trained and fused | Velociraptor |
| No test-time augmentation | **Anatomical mirror TTA** (sagittal: reverse the 3 slice channels; coronal/axial: flip width) | Raptor |
| Plain weights | **EMA** of the weights (the role SWA plays in Raptor) | Raptor |
| Fixed folds/epochs | **Presets** and a **wall-clock budget**: training stops cleanly after the last fold that fits, and the submission uses the folds finished | Dinosaur v5 deadline guards |
| Fusion weights picked on the 58 expert studies | Fusion weights fitted on **out-of-fold** predictions (shrunk toward equal); the expert set is used only for reporting | fixes a weakness seen in the public notebooks |
| Fails silently | per-study fallbacks (missing series, unreadable DICOM, no series at all -> label prevalence), checkpoint SHA-256 in `config.json` | Raptor / Blend-Gold |

Kept from v1: the multilingual report labeler (+ optional local LLM labeler), leakage-safe protocol (the 58
expert-labelled studies are never trained on unless you set `INCLUDE_GOLD=True`), soft-label BCE, bootstrap CI,
leave-one-out thresholds for accuracy.

## Honest status
* The code was smoke-tested on **synthetic** DICOMs on CPU (shapes, masks, training step, TTA, fusion,
  submission format). **No accuracy has been measured for v2.** Whatever AUC/accuracy it reaches on the real data is
  printed by the notebook; nothing here promises a number.
* Run-time and memory estimates in the presets are estimates, not measurements.
* Label quality still caps the result: in the author's study the rule labels reach ≈0.735 AUC against the expert
  set. The LLM labeler (`USE_LLM_LABELS=True`) is the largest expected lever; the better of rules / LLM /
  average is chosen with the 58 expert studies (three candidates, so a small selection effect; the 95 % CI of a
  58-study AUC is about ±0.06).
* The public notebooks' ≈0.95 scores come from checkpoints trained on LLM labels plus extra data and tuned on
  the 58 expert studies and the public leaderboard; this notebook does not reproduce them.

**Modes:** `MODE="train"` (internet ON for pretrained weights) trains, evaluates and writes weights + a submission.
`MODE="infer"` (code-competition, internet OFF): set `WEIGHTS_DIR` to a Kaggle dataset holding `config.json` and the
`*_fold*.pt` files.

In [ ]:
# ============================== CONFIG ==============================
import os, glob, json, math, time, random, warnings, hashlib, shutil
warnings.filterwarnings("ignore")
BUILD = "2026-10-09 e: pre-flight, offline weights, auto time-fit, step logging"
print("notebook build:", BUILD)

class CFG:
    MODE = "train"                 # "train" (train+eval+submit) | "infer" (load WEIGHTS_DIR, submit only)
    PRESET = "ultra"            # "fast" | "balanced" | "max" | "custom" (keep the values below)
    SEED = 42
    AUTO_FIT_TIME = True           # benchmark a few training steps, then shrink epochs / folds so training fits the budget
    MIN_EPOCHS = 4                 # the fitter keeps at least this many epochs per fold before it drops folds
    LOG_EVERY = 50                 # print step speed, ETA, host RAM and GPU memory every N steps
    TIME_BUDGET_H = 8.5            # stop starting new folds when the next one would not finish inside this budget
    INFER_RESERVE_MIN = 25         # time kept for test inference
    # ---- data ----
    DATA_DIR = None                # None -> auto-detect under /kaggle/input
    CACHE_DIR = None               # None -> the writable place with the most free disk among /kaggle/temp, /tmp, /kaggle/working
    OUT_DIR = "/kaggle/working"
    WEIGHTS_DIR = "/kaggle/input/knee-v2-weights"       # used when MODE == "infer"
    AUTO_FIT_DISK = True           # lower IMG automatically if the cache would not fit on disk
    # ---- input geometry (physical) ----
    # (plane, fluid_sensitive preference 1/0/-1=any, slices)  ->  96 slices, as in the Raptor notebooks
    SLOTS = [("Sagittal", 1, 26), ("Sagittal", 0, 22), ("Coronal", 1, 18), ("Coronal", 0, 12), ("Axial", -1, 18)]
    SPAN = (0.02, 0.98)            # fraction of each stack that is sampled (wide: collaterals / lateral meniscus)
    CROP_MM = 140.0                # square field of view around the image centre
    IMG = 256                      # cached in-plane size
    PCT = (2.0, 98.0)              # per-series intensity window (percentiles of the sampled slices)
    ALLOW_SLOT_REUSE = True        # a slot with no unused series of its plane reuses that plane's best series
    # ---- model ----
    BACKBONES = ["convnext_tiny.in12k_ft_in1k"]  # >1 entry = several arms fused on OOF; e.g. add "tf_efficientnet_b3.ns_jft_in1k"
    PRETRAINED = True              # needs internet in MODE="train"
    D_MODEL = 384
    TF_DEPTH = 2
    DROP = 0.2
    DROP_PATH = 0.1
    GRAD_CKPT = False              # gradient checkpointing in the encoder (saves memory, ~30 % slower)
    # ---- training ----
    FOLDS = 5
    TRAIN_FOLDS = [0, 1, 2, 3, 4]
    EPOCHS = 6
    BATCH = 2                      # studies per step
    TRAIN_WINDOWS = 32             # windows (3-slice stacks) sampled per study per step
    EVAL_STRIDE = 1                # 1 = every slice is a window centre at inference
    LR_BACKBONE = 1e-4
    LR_HEAD = 1e-3
    WD = 0.02
    AUX_W = 0.5                    # deep supervision of the global and the local branch
    EMA = 0.998                    # 0 = off; EMA weights are evaluated and saved
    CLIP = 2.0
    MIRROR_AUG = False             # also train on anatomically mirrored studies (Raptor uses mirror only at test time)
    MIRROR_TTA = True
    NUM_WORKERS = 4
    AMP = True
    INCLUDE_GOLD = False           # True: add the 58 expert studies to training (final-fit mode; gold metrics are then invalid)
    # ---- labels ----
    USE_LLM_LABELS = False         # True -> attach a Qwen2.5-Instruct model and set LLM_PATH
    LLM_PATH = "/kaggle/input/qwen2.5/transformers/3b-instruct/1"
    LLM_BATCH = 8
    # ---- output ----
    OUTPUT = "prob"                # "prob" | "rank" (per-column percentile rank of the ensemble)
    # ---- offline / standalone ----
    OFFLINE = None                 # None = auto-detect (no internet -> HF_HUB_OFFLINE=1); everything must then come from /kaggle/input
    BACKBONE_WEIGHTS = {}          # optional explicit {backbone_tag: "/kaggle/input/.../model.safetensors"}
    WEIGHT_SEARCH_ROOTS = ["/kaggle/input"]
    ALLOW_RANDOM_INIT = False      # True: if no pretrained file is found offline, train from random weights (much worse)
    EXPORT_BACKBONES = False       # run ONCE with internet on: saves backbone weights to OUT_DIR/backbone_weights, then stops
    RESUME_DIR = None              # a previous session's output directory (attach it as a dataset) to continue training
    SMOKE = False
    MAX_TRAIN_STUDIES = None       # e.g. 300 for a quick trial

PRESETS = {   # ESTIMATES for a Kaggle T4x2, not measurements
    "fast":     dict(IMG=224, TRAIN_WINDOWS=24, EPOCHS=4, BACKBONES=["convnext_nano.in12k_ft_in1k"], D_MODEL=256),
    "balanced": dict(),
    "ultra":    dict(IMG=384, TRAIN_WINDOWS=48, EPOCHS=8, GRAD_CKPT=True, D_MODEL=512, TF_DEPTH=3, EMA=0.999, LR_BACKBONE=8e-5,
                     USE_LLM_LABELS=True, TIME_BUDGET_H=8.6,
                     BACKBONES=["coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "convnext_small.in12k_ft_in1k"]),
    "max":      dict(IMG=320, TRAIN_WINDOWS=48, EPOCHS=8, GRAD_CKPT=True,
                     BACKBONES=["convnext_tiny.in12k_ft_in1k", "tf_efficientnet_b3.ns_jft_in1k"]),
}
for k, v in PRESETS.get(CFG.PRESET, {}).items(): setattr(CFG, k, v)

if os.environ.get("KNEE_SMOKE") == "1":   # used only by the author's CPU smoke test on synthetic data
    CFG.SMOKE, CFG.DATA_DIR = True, os.environ["KNEE_DATA_DIR"]
    CFG.CACHE_DIR = CFG.OUT_DIR = os.environ["KNEE_OUT_DIR"]
    CFG.BACKBONES, CFG.PRETRAINED, CFG.IMG, CFG.CROP_MM, CFG.D_MODEL = ["resnet18", "resnet10t"], False, 64, 100.0, 64
    CFG.SLOTS = [("Sagittal", 1, 5), ("Sagittal", 0, 4), ("Coronal", 1, 4), ("Coronal", 0, 3), ("Axial", -1, 4)]
    CFG.FOLDS, CFG.TRAIN_FOLDS, CFG.EPOCHS, CFG.BATCH, CFG.TRAIN_WINDOWS = 2, [0, 1], 1, 2, 8
    CFG.NUM_WORKERS, CFG.AMP, CFG.EMA, CFG.TF_DEPTH, CFG.AUTO_FIT_DISK = 0, False, 0.9, 1, False
    CFG.MODE = os.environ.get("KNEE_MODE", "train"); CFG.WEIGHTS_DIR = os.environ.get("KNEE_WEIGHTS", CFG.OUT_DIR)
CFG.TOTAL = sum(s[2] for s in CFG.SLOTS)
if CFG.CACHE_DIR is None:
    _opts = []
    for _d in ("/kaggle/temp", "/tmp", "/kaggle/working"):
        try:
            os.makedirs(_d, exist_ok=True); _opts.append((shutil.disk_usage(_d).free, _d))
        except Exception: pass
    print("free disk:", {d: f"{f / 1e9:.0f} GB" for f, d in _opts})
    CFG.CACHE_DIR = os.path.join(max(_opts)[1], "cache")
os.makedirs(CFG.CACHE_DIR, exist_ok=True); os.makedirs(CFG.OUT_DIR, exist_ok=True)
print({k: v for k, v in vars(CFG).items() if not k.startswith("_")})

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import cv2, pydicom, timm
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU = torch.cuda.device_count()
print("device:", DEVICE, "| GPUs:", N_GPU, "| torch", torch.__version__, "| timm", timm.__version__)

def find_data_dir():
    if CFG.DATA_DIR: return CFG.DATA_DIR
    for p in sorted(glob.glob("/kaggle/input/*")) + sorted(glob.glob("/kaggle/input/*/*")) + sorted(glob.glob("/kaggle/input/*/*/*")):
        if os.path.exists(os.path.join(p, "test_series.csv")): return p
    raise FileNotFoundError("competition data not found under /kaggle/input")
DATA = find_data_dir(); print("data:", DATA)
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
T0 = time.time()
def elapsed(): return f"{(time.time() - T0) / 60:.1f} min"
def sha256(path, n=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while b := f.read(n): h.update(b)
    return h.hexdigest()

## 0. Offline resources and pre-flight check

Everything this notebook needs must exist **inside Kaggle**. Attach (Add Input):
1. the **competition data**;
2. for the report labeler: a **Qwen2.5-Instruct** model from *Models* (`qwen-lm/qwen2.5`, Transformers, `3b-instruct` or `7b-instruct`);
3. pretrained **backbone weights** for each name in `CFG.BACKBONES`. With internet off, timm cannot download them. Create them once
   with internet on: set `EXPORT_BACKBONES=True`, run, then *Save Version* and use the notebook output
   (`backbone_weights/*.safetensors`) as a Kaggle dataset, and attach that dataset here. The notebook finds the files by name.

The check below fails **before** the hours-long DICOM caching if something is missing.

In [ ]:
import socket
def _online(host="huggingface.co", port=443, t=3):
    try:
        socket.create_connection((host, port), timeout=t).close(); return True
    except Exception:
        return False
OFFLINE = (not _online()) if CFG.OFFLINE is None else bool(CFG.OFFLINE)
if OFFLINE:
    os.environ["HF_HUB_OFFLINE"] = os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("internet:", "OFF -> using only /kaggle/input" if OFFLINE else "ON")

WEIGHT_EXT = (".safetensors", ".bin", ".pth", ".pt")
def resolve_weights(backbone):
    # explicit path > file whose path contains the full tag > file whose path contains the architecture name
    if backbone in CFG.BACKBONE_WEIGHTS: return CFG.BACKBONE_WEIGHTS[backbone]
    arch, best = backbone.split(".")[0].lower(), (0, None)
    for rt in CFG.WEIGHT_SEARCH_ROOTS:
        for root, dirs, files in os.walk(rt):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series") and "qwen" not in d.lower()]
            for fn in files:
                low = os.path.join(root, fn).lower()
                if not low.endswith(WEIGHT_EXT) or "_fold" in low or "optimizer" in low: continue
                score = 2 if backbone.lower() in low else (1 if arch in low else 0)
                path = os.path.join(root, fn)
                if score > best[0] or (score and score == best[0] and len(path) < len(best[1])): best = (score, path)
    return best[1]

def build_encoder(backbone, pretrained, kw):
    base = dict(num_classes=0, in_chans=3, **kw)
    if not pretrained: return timm.create_model(backbone, pretrained=False, **base)
    path = resolve_weights(backbone)
    if path:
        print(f"  pretrained weights for {backbone}: {path}")
        return timm.create_model(backbone, pretrained=True, pretrained_cfg_overlay=dict(file=path), **base)
    if not OFFLINE:
        return timm.create_model(backbone, pretrained=True, **base)
    if CFG.ALLOW_RANDOM_INIT:
        print(f"!! no pretrained weights for {backbone}: RANDOM INIT (expect far worse results)")
        return timm.create_model(backbone, pretrained=False, **base)
    raise FileNotFoundError(f"No pretrained weights for '{backbone}' under {CFG.WEIGHT_SEARCH_ROOTS} and internet is off. "
                            f"Set EXPORT_BACKBONES=True once with internet on, save the output as a dataset and attach it, "
                            f"or set CFG.BACKBONE_WEIGHTS['{backbone}'] to a file path.")

def resolve_llm():
    if os.path.isdir(CFG.LLM_PATH) and os.path.exists(os.path.join(CFG.LLM_PATH, "config.json")): return CFG.LLM_PATH
    cands = []
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        if "config.json" in files and "qwen" in root.lower(): cands.append(root)
    cands.sort(key=lambda r: (("instruct" not in r.lower()), len(r)))
    return cands[0] if cands else None

if CFG.EXPORT_BACKBONES:
    assert not OFFLINE, "EXPORT_BACKBONES needs internet ON (to download the weights once)"
    from safetensors.torch import save_file
    outd = f"{CFG.OUT_DIR}/backbone_weights"; os.makedirs(outd, exist_ok=True)
    for b in CFG.BACKBONES:
        kw = {"img_size": CFG.IMG} if b.startswith(("coatnet", "coat_", "maxvit", "maxxvit", "vit_", "eva", "beit")) else {}
        m = timm.create_model(b, pretrained=True, num_classes=0, **kw)
        save_file({k: v.contiguous() for k, v in m.state_dict().items()}, f"{outd}/{b}.safetensors"); print("saved", f"{outd}/{b}.safetensors")
    raise SystemExit("Backbone weights exported. Save Version, turn the output into a Kaggle dataset, attach it, set EXPORT_BACKBONES=False.")

def preflight():
    rows, ok = [], True
    rows.append(("competition data", DATA, True))
    rows.append(("GPU", f"{N_GPU} x {torch.cuda.get_device_name(0)}" if N_GPU else "none (CPU: far too slow for real runs)", N_GPU > 0 or CFG.SMOKE))
    if CFG.MODE == "train":
        for b in CFG.BACKBONES:
            if not CFG.PRETRAINED: rows.append((f"backbone {b}", "random init (PRETRAINED=False)", True)); continue
            w = resolve_weights(b); good = bool(w) or not OFFLINE or CFG.ALLOW_RANDOM_INIT
            rows.append((f"backbone {b}", w or ("download at start (internet on)" if not OFFLINE else "MISSING"), good))
        if CFG.USE_LLM_LABELS:
            q = resolve_llm(); rows.append(("Qwen LLM labeler", q or "not found -> rule labels only (warning)", True))
            if q: CFG.LLM_PATH = q
    for name, val, good in rows:
        print(f"[{'ok' if good else 'MISSING'}] {name}: {val}"); ok &= bool(good)
    free = shutil.disk_usage(CFG.CACHE_DIR).free / 1e9
    print(f"[info] free disk for the DICOM cache: {free:.0f} GB | time budget {CFG.TIME_BUDGET_H} h")
    if not ok: raise RuntimeError("Pre-flight failed: attach the missing inputs listed above (see section 0).")
preflight()

## 1. Multilingual rule labeler (embedded, unchanged from v1)

In [ ]:
%%writefile knee_labeler.py
"""Multilingual rule-based report labeler (EN/ES/DE/NL/HR/TR/BG/FR).

Only 58/4407 training studies carry expert labels; the rest ship a free-text
report in one of eight languages. This module converts reports into *silver*
labels in {0, 0.5, 1} (negative / uncertain / positive) for the 12 targets.

Design (CheXpert-labeler style, extended to multiple languages):
  1. Text normalisation: lower-case, strip Latin diacritics, map Turkish dotless i.
  2. Drop history/indication sentences and questions (they mention the
     *suspected* finding, not the observed one).
  3. Sentence -> sub-clauses (commas, contrastive conjunctions).
  4. A finding term is attributed to the *nearest* structure mention in the
     sentence (both word orders occur: "medial meniscus tear" vs
     "rotura del menisco interno" vs "medijalnog meniska ruptura").
  5. Negation / uncertainty cues are scoped to the sub-clause, plus a
     sentence-initial negation that distributes over lists ("No effusion,
     fracture or contusion").
The rules were written from the vocabulary of *unlabeled* reports; the 58 gold
studies are used only for measurement (see ``evaluate``).
"""
from __future__ import annotations

import re
import unicodedata

import numpy as np
import pandas as pd

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]


def norm(s: str) -> str:
    s = s.lower().replace("ı", "i").replace("ß", "ss")
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[ \t]+", " ", s)


def strip_accents(s: str) -> str:
    s = unicodedata.normalize("NFKD", s.replace("ı", "i"))
    return "".join(c for c in s if not unicodedata.combining(c))


def rx(*alts: str) -> re.Pattern:
    # patterns are written lower-case; only strip accents (lower() would corrupt \W, \S, ...)
    return re.compile("|".join(f"(?:{strip_accents(a)})" for a in alts))


# ----------------------------------------------------------------- cue lexicons
NEG = rx(r"\bno\b", r"\bnot\b", r"\bwithout\b", r"\bnegative\b", r"\babsent\b", r"\bfree of\b", r"\bintact\b",
         r"\bnormal", r"\bunremarkable\b", r"\bpreserved\b", r"\bruled out\b", r"\bresolved\b", r"\bnone\b",
         r"\bsin\b", r"\bni\b", r"\bintegr[oa]s?\b", r"\bconservad[oa]s?\b", r"\bausencia\b", r"\bdescart",
         r"\bkein\w*\b", r"\bohne\b", r"\bnicht\b", r"\bintakt\w*\b", r"\bunauffallig\w*\b", r"\bregelrecht\w*\b",
         r"\bgeen\b", r"\bzonder\b", r"\bniet\b", r"\bnormaal\b", r"\bgaaf\b",
         r"\bbez\b", r"\bnema\b", r"\bne\b", r"\buredn\w*\b", r"\bocuvan\w*\b", r"\bintaktn\w*\b", r"\bnije\b",
         r"\byok\w*\b", r"\bizlenme\w*\b", r"\bsaptanma\w*\b", r"\bgozlenme\w*\b", r"\bdogal\w*\b",
         r"\bolmaksizin\b", r"\bdegil\w*\b", r"\brastlanma\w*\b", r"\bizlenmemekte\w*\b",
         r"няма", r"\bбез\b", r"\bне\b", r"нормалн", r"запазен", r"интакт", r"б\.о\.", r"липсв",
         r"\bpas de\b", r"\babsence\b", r"\bsans\b", r"\bnon\b")
UNC = rx(r"\bpossibl", r"\bprobabl", r"\bsuspect", r"\bsuspicio", r"\bcannot be excluded\b", r"\blikely\b",
         r"\bmay\b", r"\bquestionable\b", r"\bequivocal\b", r"\bdd\b", r"\bposible\b", r"\bsospech",
         r"\bverdacht", r"\bfraglich", r"\bam ehesten\b", r"\bmogelijk", r"\bverdacht", r"\bwaarschijnlijk",
         r"\bsuspekt", r"\bmoguc", r"\bvjerojatn", r"\bsuphe", r"\bolasi\b", r"\bmuhtemel",
         r"вероятн", r"съмнени", r"възможн", r"\bsuspicion\b", r"\bpodria\b", r"\bkonnte\b",
         r"zou kunnen", r"\bmoze\b", r"olabilir", r"\bможе\b", r"\bpourrait\b", r"\bpodria\b")
CONTRAST = re.compile(r",|\bbut\b|\bhowever\b|\bpero\b|\bsin embargo\b|\baber\b|\bjedoch\b|\bmaar\b|\bechter\b"
                      r"|\bali\b|\bmedutim\b|\bancak\b|\bfakat\b|\bно\b|\bmais\b|\bwhereas\b|\bwhile\b")
HISTORY = rx(r"^\W*(clinical|history|indication|reason|comparison|antecedente|klinische|vraagstel|anamnes"
             r"|indikacij|klinik|клинич|renseignement|motivo|fragestellung|diagnostische)")

# ----------------------------------------------------------------- structures
S = {
    "ACL": rx(r"\bacl\b", r"\blca\b", r"anterior cruciate", r"cruzado anterior", r"\bvkb\b", r"vorder\w* kreuzband",
              r"voorste kruisband", r"prednj\w* (ukriz|kriz)\w*", r"\bon capraz", r"предн\w* кръстн",
              r"croise anterieur", r"\blia\b"),
    "PCL": rx(r"\bpcl\b", r"\blcp\b", r"posterior cruciate", r"cruzado posterior", r"\bhkb\b", r"hinter\w* kreuzband",
              r"achterste kruisband", r"straznj\w* (ukriz|kriz)\w*", r"arka capraz", r"задн\w* кръстн",
              r"croise posterieur", r"\blip\b"),
    "MCL": rx(r"\bmcl\b", r"medial collateral", r"tibial collateral", r"colateral (medial|interno)", r"\blcm\b",
              r"\blli\b", r"innenband", r"medial\w* kollateral", r"mediale collaterale", r"\bmcb\b",
              r"medijaln\w* kolateral", r"медиал\w* колатерал", r"collateral (interne|medial)"),
    "LCL": rx(r"\blcl\b", r"\bfcl\b", r"lateral collateral", r"fibular collateral", r"colateral (lateral|externo)",
              r"\blce\b", r"aussenband", r"lateral\w* kollateral", r"laterale collaterale", r"\blcb\b",
              r"lateraln\w* kolateral", r"латерал\w* колатерал", r"collateral (externe|lateral)", r"\blle\b"),
    "MedMen": rx(r"(medial|internal|interno|interna|mediale[nrs]?|medijaln\w*|медиалн\w*|interne)\W+(\w+\W+)?(menisc|menisk|menisq|менис)",
                 r"(menisc|menisk|menisq|менис)\w*\W+(\w+\W+)?(medial|interno|interna|mediale|medijaln\w*|медиалн\w*|interne)\b",
                 r"innenmenisk"),
    "LatMen": rx(r"(lateral|external|externo|externa|laterale[nrs]?|lateraln\w*|латералн\w*|externe)\W+(\w+\W+)?(menisc|menisk|menisq|менис)",
                 r"(menisc|menisk|menisq|менис)\w*\W+(\w+\W+)?(lateral|externo|externa|laterale|lateraln\w*|латералн\w*|externe)\b",
                 r"aussenmenisk"),
}
# generic meniscus mention with no side -> lets "medial and lateral menisci" etc. resolve via side words
TEAR = rx(r"\btear", r"\btorn\b", r"ruptur", r"disrupt", r"avuls", r"\brotur", r"\brot[oa]\b", r"desgarr",
          r"\briss", r"scheur", r"puknu", r"razdor", r"yirt", r"разкъс", r"руптур", r"dechir", r"flap",
          r"bucket", r"\bdefect", r"transsection", r"\bdiscontinu", r"devamsiz")
TEAR_UNC = rx(r"\blesion", r"\blasion", r"\blaesie", r"\blezij", r"\blezyon", r"увред", r"\binjur", r"ozljed", r"hasar")
SPRAIN = rx(r"sprain", r"strain", r"esguince", r"distensi", r"distorsi", r"zerrung", r"dehnung", r"verstuik",
            r"istegnu", r"burkul", r"gerilme", r"разтяг", r"entorse", r"partial", r"parcial", r"teilruptur",
            r"partie", r"parcijal", r"kismi", r"частичн", r"parziel", r"thicken", r"engros", r"verdick", r"zadeb",
            r"kalinlas", r"удебел", r"edema", r"odem", r"oedeem", r"едем", r"oedeme")

OA = rx(r"osteoart", r"osteophyt", r"osteofit", r"остеофит", r"arthros", r"artros", r"artroz", r"gonart", r"остеоарт", r"артроз", r"\boa\b",
        r"degenerative joint", r"cambios degenerativos articulares")
COMP = {
    "Medial OA": rx(r"\bmedial", r"\binterno\b", r"\binterna\b", r"\binterne\b", r"\bmediale", r"\bmedijaln",
                    r"медиалн", r"\binnen"),
    "Lateral OA": rx(r"\blateral", r"\bexterno\b", r"\bexterna\b", r"\bexterne\b", r"\blaterale", r"\blateraln",
                     r"латерал", r"\baussen"),
    "PF OA": rx(r"patel\w*femor", r"femor\w*patel", r"retropatell", r"пател\w*фемор", r"фемор\w*пател",
                r"\bpf\b", r"patellar", r"\bpatel"),
}
TRI = rx(r"tricompart", r"three compart", r"all (three )?compart", r"3 compart", r"todos los compartimentos",
         r"tricompartimental", r"drei kompart", r"alle (drie )?compartiment", r"sva tri", r"svim odjelj",
         r"tum kompartman", r"uc kompartman", r"трите", r"три компарт", r"trois compart", r"pangonart",
         r"pan-?compart")

SIMPLE = {
    "Effusion": rx(r"effusion", r"derrame", r"erguss", r"hydrops", r"effusie", r"vocht in (het )?gewricht",
                   r"gewrichtsvocht", r"izljev", r"efuzyon", r"(mayi|sivi) artis", r"излив", r"epanchement",
                   r"versamento", r"intraarticular fluid", r"joint fluid"),
    "Synovitis": rx(r"synovit", r"sinovit", r"synovialit", r"синовит", r"synovial (thickening|proliferation|hypertroph)"),
    "Baker's": rx(r"baker", r"popliteal cyst", r"quiste popliteo", r"poplitealzyste", r"popliteacyste",
                  r"poplitealn\w* cist", r"popliteal kist", r"поплит\w* кист", r"kyste poplite",
                  r"gastrocnemi\w*[- ]semimembran\w* burs", r"semimembranos\w*[- ]gastrocnem\w* burs"),
    "Contusion": rx(r"contusi", r"kontusion", r"kontuzij", r"kontuzyon", r"контузи", r"bone bruise", r"kneuzing",
                    r"bone bruising", r"\bbruise"),
    "Fracture": rx(r"fractur", r"fractuur", r"fraktur", r"prijelom", r"\bkirik", r"фрактур", r"счупв", r"breuk",
                   r"\bfx\b"),
}
BME = rx(r"bone marrow (o)?edema", r"marrow (o)?edema", r"edema oseo", r"edema de la medula", r"knochenmark\w*odem",
         r"knochenodem", r"beenmerg\w*oedeem", r"kostan\w* edem", r"kemik iligi odem", r"костно\w* едем",
         r"oedeme (osseux|medullaire|de la moelle)", r"medular\w* edem", r"bone edema")
EFF_TRIVIAL = rx(r"physiolog", r"fisiolog", r"fysiolog", r"fizioloski", r"fizyolojik", r"физиолог", r"trace",
                 r"\bminimal", r"\bminimo\b", r"\bminim\w*", r"\bgering", r"\bminieme", r"\bdiscret")


def sentences(text: str) -> list[str]:
    t = norm(text)
    parts = re.split(r"(?<![0-9])[.;\n]+(?![0-9])|\s{3,}", t)
    return [p.strip() for p in parts if p and p.strip()
            and "?" not in p and not HISTORY.search(p.strip())]


def clauses(sent: str) -> list[tuple[int, int]]:
    bounds, last = [], 0
    for m in CONTRAST.finditer(sent):
        bounds.append((last, m.start()))
        last = m.end()
    bounds.append((last, len(sent)))
    return bounds


def polarity(sent: str, pos: int, lead_neg: bool) -> float:
    """1 = affirmed, 0.5 = uncertain, 0 = negated, for the clause containing `pos`."""
    for a, b in clauses(sent):
        if a <= pos <= b:
            c = sent[a:b]
            if NEG.search(c):
                return 0.0
            if lead_neg and a > 0 and not re.search(r"\w{3,}\s+\w{3,}\s+\w{3,}\s+\w{3,}", c):
                return 0.0  # short list item after a leading "No ..." distributes the negation
            if UNC.search(c):
                return 0.5
            return 1.0
    return 1.0


def nearest(sent: str, pos: int, pats: dict[str, re.Pattern]) -> str | None:
    best, bd = None, 10 ** 9
    for k, p in pats.items():
        for m in p.finditer(sent):
            d = pos - m.end() if m.end() <= pos else (m.start() - pos) * 1.5  # prefer preceding mention
            if d < bd:
                best, bd = k, d
    return best if bd < 250 else None


def label_report(text: str) -> dict[str, float]:
    out = {k: 0.0 for k in LABELS}
    if not isinstance(text, str):
        return out
    struct_map = {"ACL": "ACL", "MCL": "MCL", "MedMen": "Medial Meniscus", "LatMen": "Lateral Meniscus"}
    for s in sentences(text):
        lead_neg = bool(re.match(r"^\W*(no|sin|kein\w*|geen|bez|nema|няма|pas de|absence of|without)\b", s))
        # --- structure injuries
        for pat, ligament_only in ((TEAR, False), (TEAR_UNC, False), (SPRAIN, True)):
            for m in pat.finditer(s):
                st = nearest(s, m.start(), S)
                if st not in struct_map or (ligament_only and st not in ("ACL", "MCL")):
                    continue
                v = polarity(s, m.start(), lead_neg)
                if pat is TEAR_UNC or (ligament_only and pat is SPRAIN and v == 1.0 and st == "ACL"):
                    v = min(v, 0.5)
                out[struct_map[st]] = max(out[struct_map[st]], v)
        # --- osteoarthritis by compartment
        for m in OA.finditer(s):
            v = polarity(s, m.start(), lead_neg)
            if v == 0:
                continue
            if TRI.search(s):
                locs = list(COMP)
            else:
                a, b = next((a, b) for a, b in clauses(s) if a <= m.start() <= b)
                window = s[max(0, a - 80):min(len(s), b + 80)]
                locs = [k for k, p in COMP.items() if p.search(window)] or \
                       ([nearest(s, m.start(), COMP)] if nearest(s, m.start(), COMP) else [])
            for k in locs:
                out[k] = max(out[k], v)
        if TRI.search(s) and re.search(r"degenera|chondro|cartilag|cartilag|knorpel|kraakbeen|hrskav|kikirda|хрущ", s):
            for k in COMP:
                out[k] = max(out[k], 0.5)
        # --- simple findings
        for lab, pat in SIMPLE.items():
            for m in pat.finditer(s):
                v = polarity(s, m.start(), lead_neg)
                if lab == "Effusion" and v > 0 and EFF_TRIVIAL.search(s):
                    v = 0.5 if not re.search(r"physiolog|fisiolog|fysiolog|fiziolo|fizyolo|физиолог", s) else 0.0
                out[lab] = max(out[lab], v)
        for m in BME.finditer(s):
            v = polarity(s, m.start(), lead_neg)
            if v > 0 and not re.search(r"subchondr|degenera|reactiv|stress|subkondr|subhondr|субхондр", s):
                out["Contusion"] = max(out["Contusion"], 0.5 * v)
    return out


def label_frame(df: pd.DataFrame) -> pd.DataFrame:
    rows = [label_report(t) for t in df["Report"].tolist()]
    res = pd.DataFrame(rows, columns=LABELS)
    res.insert(0, "StudyInstanceUID", df["StudyInstanceUID"].values)
    return res


def evaluate(gold: pd.DataFrame, pred: pd.DataFrame) -> pd.DataFrame:
    from sklearn.metrics import roc_auc_score
    g = gold.set_index("StudyInstanceUID")[LABELS]
    p = pred.set_index("StudyInstanceUID").loc[g.index, LABELS]
    rows = []
    for k in LABELS:
        y, s = g[k].values.astype(int), p[k].values
        yhat = (s >= 0.5).astype(int)
        tp, tn = int(((yhat == 1) & (y == 1)).sum()), int(((yhat == 0) & (y == 0)).sum())
        fp, fn = int(((yhat == 1) & (y == 0)).sum()), int(((yhat == 0) & (y == 1)).sum())
        rows.append(dict(label=k, n_pos=int(y.sum()), acc=(tp + tn) / len(y),
                         sens=tp / max(tp + fn, 1), spec=tn / max(tn + fp, 1),
                         auc=roc_auc_score(y, s) if 0 < y.sum() < len(y) else np.nan, tp=tp, fp=fp, fn=fn, tn=tn))
    return pd.DataFrame(rows)


if __name__ == "__main__":
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--train", default="/home/user/data/meta/train.csv")
    ap.add_argument("--out", default="/home/user/data/meta/silver_labels.csv")
    ap.add_argument("--report", default="results/report_labeler_gold_eval.csv")
    a = ap.parse_args()
    df = pd.read_csv(a.train)
    pred = label_frame(df)
    pred.to_csv(a.out, index=False)
    gold = df[df[LABELS].notna().all(axis=1)]
    ev = evaluate(gold, pred)
    ev.to_csv(a.report, index=False)
    print(ev.round(3).to_string(index=False))
    print("macro AUC vs gold: %.4f  macro acc: %.4f" % (ev.auc.mean(), ev.acc.mean()))
    print("silver prevalence (>=0.5):")
    print((pred[LABELS] >= 0.5).mean().round(3).to_string())


In [ ]:
import importlib, knee_labeler; importlib.reload(knee_labeler)
from knee_labeler import label_frame, evaluate as eval_labeler

def macro_auc(y, p, valid=None):
    a = []
    for k in range(y.shape[1]):
        v = np.ones(len(y), bool) if valid is None else valid[:, k]
        if 0 < y[v, k].sum() < v.sum(): a.append(roc_auc_score(y[v, k], p[v, k]))
    return float(np.mean(a)) if a else float("nan")

if CFG.MODE == "train":
    train = pd.read_csv(f"{DATA}/train.csv")
    gold_mask = train[LABELS].notna().all(axis=1).values
    print(f"studies: {len(train)} | expert-labelled: {gold_mask.sum()} "
          f"({'used for training' if CFG.INCLUDE_GOLD else 'held out'})")
    rule = label_frame(train).set_index("StudyInstanceUID").loc[train.StudyInstanceUID, LABELS].values
    ev = eval_labeler(train[gold_mask], pd.DataFrame(rule[gold_mask], columns=LABELS)
                      .assign(StudyInstanceUID=train.StudyInstanceUID[gold_mask].values))
    print(f"rule labeler vs expert labels: macro AUC {ev.auc.mean():.4f}")
    display(ev[["label", "n_pos", "auc", "sens", "spec"]].round(3))

## 2. Optional LLM labeler (the largest expected lever)

Set `USE_LLM_LABELS=True` and attach an instruction model (e.g. **Qwen2.5-3B/7B-Instruct** from Kaggle Models).
The model returns a probability per finding (hedged wording -> intermediate values, as in the public notebooks'
soft labels). Rules, LLM and their average are each scored on the 58 expert studies; the best one becomes the
training label. This is the only decision taken with the expert set.

In [ ]:
LLM_PROMPT = '''You are a musculoskeletal radiologist. Read the knee MRI report (any language) and rate each finding
as present in THIS knee on this exam. Answer ONLY with JSON, values between 0 and 1 (1 = definitely present,
0 = absent/not mentioned, 0.5 = equivocal, about 0.8 = suspected). Definitions:
ACL: ACL tear (partial or complete). MCL: MCL sprain/tear (any grade). Medial Meniscus / Lateral Meniscus: meniscal tear
(not degeneration alone). Medial OA / Lateral OA / PF OA: osteoarthritis of that compartment (cartilage loss,
osteophytes, degenerative joint disease). Effusion: joint effusion more than physiological. Synovitis: synovitis or
synovial thickening. Baker's: Baker/popliteal cyst. Contusion: bone contusion/bruise/traumatic marrow edema.
Fracture: any fracture.
Keys: "ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture".
REPORT:
'''

def llm_label(reports):
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(CFG.LLM_PATH, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(CFG.LLM_PATH, torch_dtype=torch.float16, device_map="auto").eval()
    out = np.full((len(reports), len(LABELS)), np.nan, np.float32)
    for i in range(0, len(reports), CFG.LLM_BATCH):
        chunk = reports[i:i + CFG.LLM_BATCH]
        msgs = [tok.apply_chat_template([{"role": "user", "content": LLM_PROMPT + str(r)[:6000]}],
                                        tokenize=False, add_generation_prompt=True) for r in chunk]
        enc = tok(msgs, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=160, do_sample=False)
        for j, g in enumerate(gen):
            txt = tok.decode(g[enc["input_ids"].shape[1]:], skip_special_tokens=True)
            try:
                js = json.loads(txt[txt.index("{"): txt.rindex("}") + 1])
                out[i + j] = [float(js.get(k, np.nan)) for k in LABELS]
            except Exception:
                pass
        if i % (CFG.LLM_BATCH * 50) == 0: print(f"  LLM {i}/{len(reports)} {elapsed()}")
    del model; torch.cuda.empty_cache()
    return out

if CFG.MODE == "train":
    candidates = {"rules": rule}
    if CFG.USE_LLM_LABELS and not os.path.isdir(CFG.LLM_PATH):
        print(f"!! USE_LLM_LABELS=True but no model at {CFG.LLM_PATH}: attach a Qwen2.5-Instruct model; continuing with rule labels")
    elif CFG.USE_LLM_LABELS:
        cached = [q for q in (f"{CFG.RESUME_DIR}/llm_labels.npy" if CFG.RESUME_DIR else None, f"{CFG.OUT_DIR}/llm_labels.npy") if q and os.path.exists(q)]
        if cached: llm = np.load(cached[0]); print("LLM labels loaded from", cached[0])
        else: llm = llm_label(train.Report.tolist()); np.save(f"{CFG.OUT_DIR}/llm_labels.npy", llm)
        ok = ~np.isnan(llm).any(1); print(f"LLM parsed {ok.mean():.1%} of reports")
        llm = np.where(np.isnan(llm), rule, np.clip(llm, 0, 1))
        candidates["llm"] = llm
        candidates["llm+rules"] = (llm + rule) / 2
    Yg_all = train.loc[gold_mask, LABELS].values.astype(int)
    scores = {k: macro_auc(Yg_all, v[gold_mask]) for k, v in candidates.items()}
    print("labeler macro AUC vs expert labels:", {k: round(v, 4) for k, v in scores.items()})
    LABEL_SOURCE = max(scores, key=scores.get)
    SOFT = candidates[LABEL_SOURCE].astype(np.float32)
    print("-> training labels from:", LABEL_SOURCE)

## 3. DICOM → physically normalised slot volumes

Per study the cache holds `TOTAL` slices (96 by default) in fixed slots. For each slot:
1. choose a series of the slot's plane, preferring the slot's fluid-sensitivity (a series is used once; if none is
   left and `ALLOW_SLOT_REUSE`, the plane's best series is reused, so the slot is not empty);
2. order the files along the slice normal (`ImageOrientationPatient` × `ImagePositionPatient`);
3. take slices evenly from 2 % to 98 % of the stack;
4. apply the modality LUT (slope/intercept), invert MONOCHROME1, clip to the series' 2–98 percentiles;
5. centre-crop a `CROP_MM` square using `PixelSpacing` (a knee covers the same fraction of the frame at 0.3 and
   0.5 mm/px), resize to `IMG` with area interpolation; store uint8.

In [ ]:
def pick_series(rows, plane, fluid, used):
    cands = [r for r in rows if r["Anatomical_Plane"] == plane]
    new = [r for r in cands if r["SeriesInstanceUID"] not in used]
    if fluid in (0, 1):
        pref = [r for r in new if int(r.get("Fluid_Sensitive", 0) or 0) == fluid]
        if pref: return pref[0], False
    if new: return new[0], False
    if CFG.ALLOW_SLOT_REUSE and cands:
        best = sorted(cands, key=lambda r: -(int(r.get("Fluid_Sensitive", 0) or 0) == fluid) if fluid in (0, 1) else 0)
        return best[0], True
    return None, False

def _hdr(f):
    try:
        ds = pydicom.dcmread(f, stop_before_pixels=True, force=True)
        o, p = ds.get("ImageOrientationPatient"), ds.get("ImagePositionPatient")
        if o is not None and p is not None and len(o) == 6:
            o = np.array(o, float); pos = float(np.dot(np.cross(o[:3], o[3:]), np.array(p, float)))
        else:
            pos = float(ds.get("InstanceNumber", 0) or 0)
        ps = ds.get("PixelSpacing"); ps = float(ps[0]) if ps is not None else 0.0
        return pos, (int(ds.Rows), int(ds.Columns)), ps, f
    except Exception:
        return None

def _read_px(f):
    ds = pydicom.dcmread(f, force=True)
    a = ds.pixel_array
    if a.ndim != 2: raise ValueError("multi-frame / colour")
    a = a.astype(np.float32)
    s, i = ds.get("RescaleSlope"), ds.get("RescaleIntercept")
    if s is not None and i is not None: a = a * float(s) + float(i)
    if str(ds.get("PhotometricInterpretation", "")) == "MONOCHROME1": a = a.max() - a
    return a

def mm_crop_resize(a, ps, size):
    h, w = a.shape
    if ps and ps > 0:
        c = int(round(CFG.CROP_MM / ps)); ch, cw = min(c, h), min(c, w)
    else:                                   # unknown spacing: assume the image spans about the field of view
        ch = cw = min(h, w)
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    crop = np.ascontiguousarray(a[y0:y0 + ch, x0:x0 + cw])
    out = cv2.resize(crop, (size, size), interpolation=cv2.INTER_AREA)
    return out

def series_slices(files, n, size, threads=8):
    with ThreadPoolExecutor(threads) as ex:
        hdr = [h for h in ex.map(_hdr, files) if h is not None]
    if not hdr: return None
    shp = pd.Series([h[1] for h in hdr]).value_counts().index[0]
    hdr = sorted([h for h in hdr if h[1] == shp], key=lambda h: h[0])
    ps_all = [h[2] for h in hdr if h[2] > 0]; ps = float(np.median(ps_all)) if ps_all else 0.0
    m = len(hdr); lo, hi = int(m * CFG.SPAN[0]), max(int(m * CFG.SPAN[1]) - 1, int(m * CFG.SPAN[0]))
    idx = np.linspace(lo, hi, n).round().astype(int) if m > 1 else np.zeros(n, int)
    uniq = sorted(set(idx.tolist()))
    def rd(i):
        try: return _read_px(hdr[i][3])
        except Exception: return None
    with ThreadPoolExecutor(threads) as ex:
        px = dict(zip(uniq, ex.map(rd, uniq)))
    good = [a for a in px.values() if a is not None]
    if not good: return None
    samp = np.concatenate([a[::4, ::4].ravel() for a in good])
    lo_v, hi_v = np.percentile(samp, CFG.PCT)
    out = []
    for i in idx:
        a = px[int(i)]
        a = np.zeros(shp, np.float32) if a is None else np.clip((a - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)
        out.append(mm_crop_resize(a, ps, size))
    return (np.stack(out) * 255).round().astype(np.uint8)

def study_volume(args):
    split, st, rows = args
    vol = np.zeros((CFG.TOTAL, CFG.IMG, CFG.IMG), np.uint8); smask = np.zeros(len(CFG.SLOTS), bool)
    used, pos = set(), 0
    for s, (plane, fluid, n) in enumerate(CFG.SLOTS):
        try:
            r, reused = pick_series(rows, plane, fluid, used)
            if r is not None:
                used.add(r["SeriesInstanceUID"])
                v = series_slices(glob.glob(f"{DATA}/{split}_series/{st}/{r['SeriesInstanceUID']}/*.dcm"), n, CFG.IMG)
                if v is not None: vol[pos:pos + n] = v; smask[s] = True
        except Exception as e:
            print(f"  [cache] {st[:12]} slot {s} failed: {type(e).__name__}: {str(e)[:80]}")
        pos += n
    return vol, smask

def plan_cache(n_train, n_test):
    global_img = CFG.IMG
    free = shutil.disk_usage(CFG.CACHE_DIR).free
    need = lambda img: (n_train + n_test) * CFG.TOTAL * img * img
    while CFG.AUTO_FIT_DISK and need(global_img) > 0.9 * free and global_img > 160:
        global_img -= 32
    if global_img != CFG.IMG:
        print(f"!! cache would need {need(CFG.IMG) / 1e9:.1f} GB, {free / 1e9:.1f} GB free -> IMG {CFG.IMG} -> {global_img}")
        CFG.IMG = global_img
    print(f"cache estimate: {need(CFG.IMG) / 1e9:.1f} GB at IMG={CFG.IMG} ({free / 1e9:.1f} GB free)")

def build_cache(split, studies):
    sig = hashlib.md5(json.dumps([CFG.SLOTS, CFG.SPAN, CFG.CROP_MM, CFG.IMG, CFG.PCT, CFG.ALLOW_SLOT_REUSE, len(studies)]).encode()).hexdigest()[:10]
    vp, mp = f"{CFG.CACHE_DIR}/{split}_{sig}_vol.npy", f"{CFG.CACHE_DIR}/{split}_{sig}_mask.npy"
    if os.path.exists(mp): return np.load(vp, mmap_mode="r"), np.load(mp)
    sdf = pd.read_csv(f"{DATA}/{split}_series.csv")
    g = {s: d.to_dict("records") for s, d in sdf.groupby("StudyInstanceUID")}
    vol = np.lib.format.open_memmap(vp, "w+", np.uint8, (len(studies), CFG.TOTAL, CFG.IMG, CFG.IMG))
    mask = np.zeros((len(studies), len(CFG.SLOTS)), bool)
    jobs = [(split, s, g.get(s, [])) for s in studies]
    t = time.time()
    if CFG.NUM_WORKERS == 0: it = map(study_volume, jobs)
    else: ex = ProcessPoolExecutor(max(1, os.cpu_count() or 1)); it = ex.map(study_volume, jobs, chunksize=2)
    for i, (v, m) in enumerate(it):
        vol[i], mask[i] = v, m
        if (i + 1) % 250 == 0: print(f"  {split}: {i + 1}/{len(studies)}  {time.time() - t:.0f}s")
    vol.flush(); np.save(mp, mask)
    print(f"cached {split}: {len(studies)} studies in {time.time() - t:.0f}s; slot coverage {mask.mean(0).round(3)}; "
          f"studies with no usable series: {int((mask.sum(1) == 0).sum())}")
    return np.load(vp, mmap_mode="r"), mask

if CFG.MODE == "train":
    if CFG.MAX_TRAIN_STUDIES:
        keep = sorted(np.where(gold_mask)[0].tolist() + [i for i in range(len(train)) if not gold_mask[i]][:CFG.MAX_TRAIN_STUDIES])
        train, rule, SOFT, gold_mask = train.iloc[keep].reset_index(drop=True), rule[keep], SOFT[keep], gold_mask[keep]
    studies = train.StudyInstanceUID.tolist()
    n_test = len(pd.read_csv(f"{DATA}/test.csv"))
    plan_cache(len(studies), n_test)
    VOL, MASK = build_cache("train", studies)
    print("train cache ready", elapsed())

## 4. Windows, augmentation and the model

* A **window** is 3 adjacent slices of the same slot (edges replicated, never crossing slot borders) used as RGB.
* Training samples `TRAIN_WINDOWS` windows per study (jittered even spacing over the valid windows); inference
  uses every `EVAL_STRIDE`-th slice as a centre.
* Augmentation on the GPU, shared within a slot of a study: small rotation / scale / shift, gamma and gain. **No
  naive flips** (they would swap medial/lateral). `MIRROR_AUG` applies the anatomical mirror instead.
* **Model.** timm CNN per window -> projection + slot embedding + slice-position embedding -> 2-layer transformer
  over all windows of the study -> **one attention distribution per finding** (local branch) ; a global branch
  (masked mean per slot -> linear) ; final logit = mean of both, each also supervised (deep supervision).

In [ ]:
SLOT_OF = np.concatenate([[s] * n for s, (_, _, n) in enumerate(CFG.SLOTS)])           # slice -> slot
SLOT_START = np.cumsum([0] + [n for _, _, n in CFG.SLOTS])[:-1]
SAG_SLOTS = [s for s, (p, _, _) in enumerate(CFG.SLOTS) if p == "Sagittal"]
N_SLOT = len(CFG.SLOTS)

def window_table():
    # for every slice j (a window centre): the 3 slice indices (clipped to its slot) and its relative position in the slot
    idx = np.zeros((CFG.TOTAL, 3), np.int64); pos = np.zeros(CFG.TOTAL, np.float32)
    for s, (_, _, n) in enumerate(CFG.SLOTS):
        a = SLOT_START[s]
        for k in range(n):
            idx[a + k] = [a + max(k - 1, 0), a + k, a + min(k + 1, n - 1)]
            pos[a + k] = k / max(n - 1, 1)
    return idx, pos
WIN_IDX, WIN_POS = window_table()

class KneeDS(torch.utils.data.Dataset):
    # returns uint8 windows [K,3,H,W], slot id [K], position [K], window-valid [K], slot-valid [N_SLOT]
    def __init__(self, vol, mask, idx, y=None, train=False):
        self.vol, self.mask, self.idx, self.y, self.train = vol, mask, idx, y, train
        self.K = CFG.TRAIN_WINDOWS if train else len(range(0, CFG.TOTAL, CFG.EVAL_STRIDE))
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]; sm = self.mask[j].copy()
        valid_c = np.where(sm[SLOT_OF])[0]                                   # valid window centres
        K = self.K
        if self.train and len(valid_c):
            edges = np.linspace(0, len(valid_c), K + 1)
            sel = valid_c[np.minimum((edges[:-1] + np.random.rand(K) * np.diff(edges)).astype(int), len(valid_c) - 1)]
            wv = np.ones(K, bool)
        else:
            cen = np.arange(0, CFG.TOTAL, CFG.EVAL_STRIDE)
            sel = cen[sm[SLOT_OF[cen]]] if len(valid_c) else cen[:0]
            wv = np.zeros(K, bool); wv[:len(sel)] = True
            sel = np.pad(sel, (0, K - len(sel)))
        need = np.unique(WIN_IDX[sel]) if len(valid_c) else np.array([0])
        sl = np.zeros((CFG.TOTAL, CFG.IMG, CFG.IMG), np.uint8)
        if len(valid_c): sl[need] = self.vol[j][need]
        x = torch.from_numpy(sl[WIN_IDX[sel]])                               # K,3,H,W
        x[~torch.from_numpy(wv)] = 0
        out = {"x": x, "slot": torch.from_numpy(SLOT_OF[sel]).long(), "pos": torch.from_numpy(WIN_POS[sel]),
               "wv": torch.from_numpy(wv), "sm": torch.from_numpy(sm)}
        if self.y is not None: out["y"] = torch.from_numpy(self.y[i])
        return out

def mirror(x, slot):
    # anatomical mirror of [B,K,3,H,W]: sagittal windows reverse the 3 slice channels, coronal/axial flip the width
    is_sag = torch.isin(slot, torch.tensor(SAG_SLOTS, device=slot.device))[:, :, None, None, None]
    return torch.where(is_sag, x.flip(2), x.flip(-1))

def gpu_augment(x, slot):
    # x: B,K,3,H,W float in [0,1]; parameters are drawn per (study, slot) and shared by the windows of that slot
    B, K, _, H, W = x.shape; dev = x.device
    r = lambda lo, hi, *s: torch.empty(*s, device=dev).uniform_(lo, hi)
    g, gain, bias = r(0.7, 1.4, B, N_SLOT), r(0.85, 1.15, B, N_SLOT), r(-0.08, 0.08, B, N_SLOT)
    sc, ang, tx, ty = r(0.9, 1.1, B, N_SLOT), r(-0.15, 0.15, B, N_SLOT), r(-0.08, 0.08, B, N_SLOT), r(-0.08, 0.08, B, N_SLOT)
    take = lambda p: p.gather(1, slot)                                     # B,K
    gk, ak, sk = take(g), take(ang), take(sc)
    x = (x.clamp(0, 1) ** gk[..., None, None, None]) * take(gain)[..., None, None, None] + take(bias)[..., None, None, None]
    th = torch.zeros(B * K, 2, 3, device=dev)
    th[:, 0, 0] = (sk * torch.cos(ak)).flatten(); th[:, 0, 1] = (-sk * torch.sin(ak)).flatten()
    th[:, 1, 0] = (sk * torch.sin(ak)).flatten(); th[:, 1, 1] = (sk * torch.cos(ak)).flatten()
    th[:, 0, 2], th[:, 1, 2] = take(tx).flatten(), take(ty).flatten()
    grid = F.affine_grid(th, (B * K, 3, H, W), align_corners=False)
    x = F.grid_sample(x.reshape(B * K, 3, H, W), grid, align_corners=False, padding_mode="zeros")
    return x.reshape(B, K, 3, H, W).clamp(0, 1)

class Knee25DMIL(nn.Module):
    def __init__(self, backbone, pretrained, d=384, n=12, depth=2, drop=0.2, drop_path=0.1, grad_ckpt=False):
        super().__init__()
        kw = {"drop_path_rate": drop_path} if drop_path else {}
        if backbone.startswith(("coatnet", "coat_", "maxvit", "maxxvit", "vit_", "eva", "beit")): kw["img_size"] = CFG.IMG
        self.enc = build_encoder(backbone, pretrained, kw)
        if grad_ckpt and hasattr(self.enc, "set_grad_checkpointing"): self.enc.set_grad_checkpointing(True)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1), persistent=False)
        C = self.enc.num_features
        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(C, d), nn.LayerNorm(d))
        self.slot_emb = nn.Embedding(N_SLOT, d)
        self.pos_emb = nn.Sequential(nn.Linear(1, d), nn.GELU(), nn.Linear(d, d))
        layer = nn.TransformerEncoderLayer(d, 8 if d % 8 == 0 else 4, 2 * d, 0.1, batch_first=True, norm_first=True)
        self.mix = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False) if depth else None
        self.att = nn.Sequential(nn.Linear(d, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))   # one attention per finding
        self.w = nn.Parameter(torch.randn(n, d) * 0.02); self.b = nn.Parameter(torch.zeros(n))
        self.glob = nn.Sequential(nn.LayerNorm(N_SLOT * d), nn.Dropout(drop), nn.Linear(N_SLOT * d, n))
    def forward(self, x, slot, pos, wv):                 # x uint8-or-float [B,K,3,H,W] in [0,1]
        B, K = x.shape[:2]
        flat = x.flatten(0, 1)[wv.flatten()]
        if flat.shape[0]:
            f = self.enc(((flat - self.mean) / self.std).contiguous(memory_format=torch.channels_last))
        else:                                              # a batch made only of studies without series
            f = x.new_zeros(0, self.enc.num_features)
        feat = f.new_zeros(B * K, f.shape[-1]); feat[wv.flatten()] = f
        h = self.proj(feat.float().view(B, K, -1)) + self.slot_emb(slot) + self.pos_emb(pos.unsqueeze(-1).to(feat.dtype))
        pad = ~wv
        pad = pad & ~pad.all(1, keepdim=True)            # a study with no series attends to everything (output is overridden later)
        if self.mix is not None: h = self.mix(h, src_key_padding_mask=pad)
        a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)             # B,K,12
        zl = (torch.einsum("bkc,bkd->bcd", a.to(h.dtype), h) * self.w).sum(-1) + self.b
        oh = F.one_hot(slot, N_SLOT).to(h.dtype) * wv.unsqueeze(-1).to(h.dtype)             # B,K,S
        sm = torch.einsum("bks,bkd->bsd", oh, h) / oh.sum(1).clamp_min(1).unsqueeze(-1)
        zg = self.glob(sm.flatten(1))
        return 0.5 * (zg + zl), zg, zl

def make_model(backbone, pretrained):
    return Knee25DMIL(backbone, pretrained, CFG.D_MODEL, 12, CFG.TF_DEPTH, CFG.DROP, CFG.DROP_PATH, CFG.GRAD_CKPT)

_m = make_model(CFG.BACKBONES[0], False)
print(f"{CFG.BACKBONES[0]}: {sum(p.numel() for p in _m.parameters()) / 1e6:.2f} M parameters | windows/study: train {CFG.TRAIN_WINDOWS}, eval {len(range(0, CFG.TOTAL, CFG.EVAL_STRIDE))}")
del _m

## 5. Training (expert-labelled studies held out; EMA weights; time-budget aware)

Each arm (backbone) is trained on the same stratified folds. Before each fold the notebook checks whether it still
fits inside `TIME_BUDGET_H` (using the time of the folds already done); if not, it stops and the ensemble uses the
completed folds.

In [ ]:
class EMA:
    def __init__(self, model, decay):
        self.decay, self.shadow = decay, {k: v.detach().clone().float() for k, v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point: self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)
            else: self.shadow[k].copy_(v)
    def state_dict(self, like): return {k: self.shadow[k].to(like[k].dtype) for k in like}

def forward_batch(model, b, train, mirror_flag=False):
    x = b["x"].to(DEVICE, non_blocking=True).float().div_(255); slot = b["slot"].to(DEVICE)
    pos, wv = b["pos"].to(DEVICE), b["wv"].to(DEVICE)
    if train: x = gpu_augment(x, slot)
    if mirror_flag: x = mirror(x, slot)
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=CFG.AMP and DEVICE.type == "cuda"):
        return model(x, slot, pos, wv)

def mem_report():
    out = []
    try:
        import psutil; vm = psutil.virtual_memory(); out.append(f"host RAM {vm.used / 1e9:.1f}/{vm.total / 1e9:.1f} GB")
        if vm.available < 2e9: out.append("!! LOW HOST RAM")
    except Exception: pass
    if DEVICE.type == "cuda":
        out.append("GPU peak " + "/".join(f"{torch.cuda.max_memory_allocated(i) / 1e9:.1f}" for i in range(N_GPU)) + " GB")
    return " | ".join(out)

def train_step(model, b, opt, scaler):
    y = b["y"].to(DEVICE)
    z, zg, zl = forward_batch(model, b, True, CFG.MIRROR_AUG and random.random() < 0.5)
    bce = lambda t: F.binary_cross_entropy_with_logits(t.float(), y)
    loss = bce(z) + CFG.AUX_W * (bce(zg) + bce(zl))
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward(); scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.CLIP)
    scaler.step(opt); scaler.update()
    return loss.item(), len(y)

def train_epoch(model, loader, opt, sched, scaler, ema, tag=""):
    model.train(); tot, n, t0 = 0.0, 0, time.time()
    for i, b in enumerate(loader):
        l, m = train_step(model, b, opt, scaler); sched.step()
        if ema is not None: ema.update(model.module if hasattr(model, "module") else model)
        tot += l * m; n += m
        if CFG.LOG_EVERY and (i + 1) % CFG.LOG_EVERY == 0:
            rate = (i + 1) / (time.time() - t0)
            print(f"  {tag} step {i + 1}/{len(loader)} loss {tot / n:.4f} | {rate:.2f} it/s | epoch ETA {(len(loader) - i - 1) / rate / 60:.1f} min | {mem_report()}", flush=True)
    return tot / max(n, 1)

@torch.no_grad()
def predict(model, loader, tta=None):
    tta = CFG.MIRROR_TTA if tta is None else tta
    model.eval(); out = []
    for b in loader:
        p = torch.sigmoid(forward_batch(model, b, False)[0].float())
        if tta: p = 0.5 * (p + torch.sigmoid(forward_batch(model, b, False, True)[0].float()))
        p = p.cpu().numpy(); empty = ~b["sm"].any(1).numpy()
        if empty.any(): p[empty] = np.nan                    # no usable series: filled with the prevalence by the caller
        out.append(p)
    return np.concatenate(out)

def loader(idx, y=None, train=False):
    ds = KneeDS(VOL, MASK, idx, y, train)
    return torch.utils.data.DataLoader(ds, batch_size=CFG.BATCH if train else max(1, CFG.BATCH), shuffle=train,
                                       num_workers=min(CFG.NUM_WORKERS, os.cpu_count() or 1), pin_memory=DEVICE.type == "cuda", drop_last=train,
                                       persistent_workers=train and CFG.NUM_WORKERS > 0)

def fill_nan(P, prev): return np.where(np.isnan(P), prev[None], P)

if CFG.MODE == "train":
    trainable = np.ones(len(train), bool) if CFG.INCLUDE_GOLD else ~gold_mask
    pool = np.where(trainable)[0]; gold = np.where(gold_mask)[0]
    Ypool = SOFT[pool].copy()
    if CFG.INCLUDE_GOLD:                                       # expert labels replace the report labels for the gold studies
        Ypool[gold_mask[pool]] = train.loc[gold_mask, LABELS].values.astype(np.float32)   # pool and gold rows are both in ascending order
    Yg = train.loc[gold_mask, LABELS].values.astype(int)
    prev = Ypool.mean(0)
    strat = np.clip((Ypool >= 0.5).sum(1), 0, 5) * 2 + (Ypool[:, 0] >= 0.5)
    folds = np.zeros(len(pool), int)
    for f, (_, te) in enumerate(StratifiedKFold(CFG.FOLDS, shuffle=True, random_state=CFG.SEED).split(pool, strat)):
        folds[te] = f
    # ---- measure real speed, then fit epochs / folds into the time budget (the presets' speed numbers are only estimates) ----
    def benchmark_arm(backbone, n_warm=2, n_meas=8):
        tr = pool[folds != 0][:CFG.BATCH * (n_warm + n_meas + 1)]
        model = make_model(backbone, CFG.PRETRAINED).to(DEVICE).to(memory_format=torch.channels_last)
        if N_GPU > 1: model = nn.DataParallel(model)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-6); scaler = torch.amp.GradScaler(enabled=CFG.AMP and DEVICE.type == "cuda")
        model.train(); ts = []
        for i, b in enumerate(loader(tr, Ypool[folds != 0][:len(tr)], train=True)):
            t = time.time(); train_step(model, b, opt, scaler)
            if DEVICE.type == "cuda": torch.cuda.synchronize()
            ts.append(time.time() - t)
            if len(ts) >= n_warm + n_meas: break
        r = float(np.mean(ts[n_warm:])); print(f"  benchmark {backbone}: {r:.2f} s/step (batch {CFG.BATCH} studies x {CFG.TRAIN_WINDOWS} windows) | {mem_report()}")
        del model, opt; torch.cuda.empty_cache(); return r
    if CFG.AUTO_FIT_TIME:
        step_s = sum(benchmark_arm(b) for b in CFG.BACKBONES)                      # one step of every arm
        epoch_cost = step_s * ((len(pool) - len(pool) // CFG.FOLDS) // CFG.BATCH)   # seconds for one epoch of all arms
        budget = CFG.TIME_BUDGET_H * 3600 - (time.time() - T0) - CFG.INFER_RESERVE_MIN * 60
        EVAL_EPOCH_EQ = 0.6                                                       # val + expert-set prediction with TTA, in epochs
        plan = None
        for nf in range(min(CFG.FOLDS, len(CFG.TRAIN_FOLDS)), 0, -1):
            e = int(budget / nf / epoch_cost - EVAL_EPOCH_EQ)
            if e >= CFG.MIN_EPOCHS: plan = (nf, min(e, CFG.EPOCHS)); break
        fits = plan is not None
        if plan is None: plan = (1, max(1, min(CFG.EPOCHS, int(budget / epoch_cost - EVAL_EPOCH_EQ))))
        print(f"time plan: one epoch of all arms = {epoch_cost / 60:.1f} min; budget {budget / 3600:.1f} h -> {plan[0]} fold(s) x {plan[1]} epoch(s) "
              f"(requested {len(CFG.TRAIN_FOLDS)} x {CFG.EPOCHS})")
        if not fits: print("!! even one fold cannot get MIN_EPOCHS in this budget: lower IMG / TRAIN_WINDOWS / use a smaller backbone (preset 'fast')")
        CFG.TRAIN_FOLDS, CFG.EPOCHS = list(CFG.TRAIN_FOLDS)[:plan[0]], plan[1]
    ARMS = {f"a{i}": dict(backbone=b, oof=np.full(Ypool.shape, np.nan, np.float32), gold=[], files=[]) for i, b in enumerate(CFG.BACKBONES)}
    RES = CFG.RESUME_DIR if CFG.RESUME_DIR and os.path.isdir(CFG.RESUME_DIR) else None
    fold_sig = hashlib.md5(folds.tobytes() + np.asarray(pool).tobytes()).hexdigest()
    fold_times, stop = [], False
    for f in CFG.TRAIN_FOLDS:                                   # fold-major: every finished fold contains all arms
        left = CFG.TIME_BUDGET_H * 3600 - (time.time() - T0) - CFG.INFER_RESERVE_MIN * 60
        pending = any(not (RES and os.path.exists(f"{RES}/a{i}_fold{f}.pt")) for i in range(len(CFG.BACKBONES)))
        if pending and fold_times and left < 1.15 * np.mean(fold_times):
            print(f"!! time budget: {left / 60:.0f} min left < one fold ({np.mean(fold_times) / 60:.0f} min); stopping. "
                  f"Run again with RESUME_DIR set to this output to train the remaining folds."); break
        t_fold, trained = time.time(), False
        for arm_i, backbone in enumerate(CFG.BACKBONES):
            arm = f"a{arm_i}"; st = f"{arm}_fold{f}"; va = pool[folds == f]
            if RES and os.path.exists(f"{RES}/{st}.pt") and os.path.exists(f"{RES}/{st}_state.npz"):
                z = np.load(f"{RES}/{st}_state.npz")
                if str(z["sig"]) == fold_sig:
                    for ext in (".pt", "_state.npz"):
                        if os.path.abspath(RES) != os.path.abspath(CFG.OUT_DIR): shutil.copy(f"{RES}/{st}{ext}", f"{CFG.OUT_DIR}/{st}{ext}")
                    ARMS[arm]["oof"][folds == f] = z["oof"]
                    if not CFG.INCLUDE_GOLD: ARMS[arm]["gold"].append(z["gold"])
                    ARMS[arm]["files"].append(f"{CFG.OUT_DIR}/{st}.pt"); print("resumed", st); continue
                print(f"!! {st}: stored folds differ from this run's folds (labels/seed changed); retraining")
            trained = True; seed_all(CFG.SEED + f + 100 * arm_i)
            tr = pool[folds != f]
            model = make_model(backbone, CFG.PRETRAINED).to(DEVICE).to(memory_format=torch.channels_last)
            if N_GPU > 1: model = nn.DataParallel(model)
            core = model.module if hasattr(model, "module") else model
            enc_ids = {id(p) for p in core.enc.parameters()}
            opt = torch.optim.AdamW([{"params": list(core.enc.parameters()), "lr": CFG.LR_BACKBONE},
                                     {"params": [p for p in core.parameters() if id(p) not in enc_ids], "lr": CFG.LR_HEAD}],
                                    weight_decay=CFG.WD)
            dl = loader(tr, Ypool[folds != f], train=True)
            sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[CFG.LR_BACKBONE, CFG.LR_HEAD], total_steps=CFG.EPOCHS * len(dl), pct_start=0.1)
            scaler = torch.amp.GradScaler(enabled=CFG.AMP and DEVICE.type == "cuda")
            ema = EMA(core, CFG.EMA) if CFG.EMA else None
            for ep in range(CFG.EPOCHS):
                loss = train_epoch(model, dl, opt, sched, scaler, ema, tag=f"[{arm} f{f} ep{ep + 1}]")
                print(f"[{arm} {backbone}] fold {f} ep {ep + 1}/{CFG.EPOCHS} loss {loss:.4f}  {elapsed()}")
            if ema is not None: core.load_state_dict(ema.state_dict(core.state_dict()))   # evaluate and save the EMA weights
            pv = fill_nan(predict(model, loader(va)), prev); yv = Ypool[folds == f]
            ARMS[arm]["oof"][folds == f] = pv
            print(f"== [{arm}] fold {f}: OOF macroAUC vs report labels {macro_auc((yv >= 0.5).astype(int), pv, yv != 0.5):.4f}")
            gp = np.zeros(1)
            if not CFG.INCLUDE_GOLD:
                gp = fill_nan(predict(model, loader(gold)), prev); ARMS[arm]["gold"].append(gp)
                print(f"   expert-set macroAUC (this fold model): {macro_auc(Yg, gp):.4f}")
            fn = f"{CFG.OUT_DIR}/{st}.pt"; torch.save(core.state_dict(), fn); ARMS[arm]["files"].append(fn)
            np.savez(f"{CFG.OUT_DIR}/{st}_state.npz", oof=pv, gold=gp, sig=fold_sig)
            del model, core, opt, dl; torch.cuda.empty_cache()
        if trained: fold_times.append(time.time() - t_fold)
    cfg_dump = {k: v for k, v in vars(CFG).items() if not k.startswith("_") and isinstance(v, (int, float, str, list, tuple, bool, type(None)))}
    cfg_dump["arms"] = {a: dict(backbone=v["backbone"], files=[os.path.basename(x) for x in v["files"]],
                                sha256=[sha256(x) for x in v["files"]]) for a, v in ARMS.items()}
    cfg_dump["prevalence"] = prev.tolist()

## 6. Arm fusion (fitted on out-of-fold predictions, never on the expert set) and evaluation

* With several arms, each finding gets a weight `w` for arm 1 vs arm 2 … chosen on the **OOF** AUC against the
  report labels over a grid and **shrunk 50 % toward equal weights**, because those labels are noisy. One arm = no fusion.
* Reported, all measured by this cell: OOF macro AUC; expert-set macro AUC with a 95 % bootstrap CI; expert-set
  accuracy at leave-one-out thresholds next to the "always negative" floor; a per-label table.

In [ ]:
def rankpct(P): return (pd.DataFrame(P).rank(method="average").values - 0.5) / len(P)

def fuse_weights(arms, Y):
    # returns W[arm, label]; coordinate search over a simplex grid on OOF, then shrink toward uniform
    names = list(arms); A = len(names)
    if A == 1: return {names[0]: np.ones(12)}
    R = {a: rankpct(arms[a]["oof"]) for a in names}
    W = np.full((A, 12), 1 / A)
    grid = np.linspace(0, 1, 11)
    for k in range(12):
        v = ~np.isnan(R[names[0]][:, k]) & (Y[:, k] != 0.5); y = (Y[v, k] >= 0.5).astype(int)
        if y.sum() == 0 or y.sum() == len(y): continue
        best, bw = -1, None
        for g in (grid if A == 2 else [None]):
            w = np.array([1 - g, g]) if A == 2 else np.full(A, 1 / A)
            s = roc_auc_score(y, sum(w[i] * R[names[i]][v, k] for i in range(A)))
            if s > best: best, bw = s, w
        W[:, k] = 0.5 * bw + 0.5 / A
    return {a: W[i] for i, a in enumerate(names)}

def ensemble(parts, W, key):
    names = list(parts); return sum(W[a][None] * rankpct(parts[a]) for a in names) if key == "rank" else sum(W[a][None] * parts[a] for a in names)

def loo_accuracy(y, p):
    n, L = y.shape; hit = np.zeros((n, L), bool)
    for k in range(L):
        cand = np.unique(np.r_[p[:, k], 1.01])
        for i in range(n):
            mm = np.arange(n) != i
            t = cand[np.argmax([((p[mm, k] >= c) == y[mm, k]).mean() for c in cand])]
            hit[i, k] = (p[i, k] >= t) == y[i, k]
    return hit.mean(), hit.mean(0)

if CFG.MODE == "train":
    done_arms = {a: v for a, v in ARMS.items() if v["files"]}
    W = fuse_weights(done_arms, Ypool)
    cfg_dump["fusion_weights"] = {a: w.tolist() for a, w in W.items()}
    oof_all = {a: v["oof"] for a, v in done_arms.items()}
    done = ~np.isnan(next(iter(oof_all.values()))).any(1)
    fused_oof = ensemble({a: o[done] for a, o in oof_all.items()}, W, "rank")
    yb = (Ypool[done] >= 0.5).astype(int)
    summary = {"label_source": LABEL_SOURCE, "labeler_gold_auc": scores[LABEL_SOURCE],
               "oof_macro_auc_fused": macro_auc(yb, fused_oof, Ypool[done] != 0.5),
               "oof_macro_auc_per_arm": {a: macro_auc(yb, o[done], Ypool[done] != 0.5) for a, o in oof_all.items()},
               "folds_trained_per_arm": {a: len(v["files"]) for a, v in done_arms.items()},
               "training_minutes": (time.time() - T0) / 60, "include_gold": CFG.INCLUDE_GOLD}
    if not CFG.INCLUDE_GOLD:
        Gp = {a: np.mean(v["gold"], 0) for a, v in done_arms.items()}
        G = ensemble(Gp, W, "rank")
        rng = np.random.default_rng(0); boots = []
        for _ in range(1000):
            i = rng.integers(0, len(Yg), len(Yg)); boots.append(macro_auc(Yg[i], G[i]))
        acc, acc_lab = loo_accuracy(Yg, G)
        summary.update({"gold_macro_auc_fused": macro_auc(Yg, G), "gold_auc_95ci": list(np.nanpercentile(boots, [2.5, 97.5])),
                        "gold_macro_auc_per_arm": {a: macro_auc(Yg, g) for a, g in Gp.items()},
                        "gold_accuracy_LOO_thresholds": acc, "gold_accuracy_all_negative": float((Yg == 0).mean())})
        lab_acc = float(((candidates[LABEL_SOURCE][gold_mask] >= 0.5).astype(int) == Yg).mean())
        summary["labeler_as_predictor_gold_accuracy"] = lab_acc
        print(f"TARGET CHECK  macro accuracy (LOO thresholds) {acc:.4f} | macro AUC {summary['gold_macro_auc_fused']:.4f} | "
              f"99% target {'REACHED' if acc >= 0.99 else 'NOT reached'} | always-negative floor {summary['gold_accuracy_all_negative']:.4f} | "
              f"accuracy of the training labels themselves vs expert labels {lab_acc:.4f} (what the model is taught from)")
        per = pd.DataFrame({"gold_auc": [macro_auc(Yg[:, k:k + 1], G[:, k:k + 1]) for k in range(12)], "gold_acc_LOO": acc_lab,
                            "oof_auc": [macro_auc(yb[:, k:k + 1], fused_oof[:, k:k + 1], Ypool[done, k:k + 1] != 0.5) for k in range(12)]},
                           index=LABELS).round(4)
    else:
        print("INCLUDE_GOLD=True: the expert studies were trained on, so no expert-set metric is reported.")
        per = pd.DataFrame({"oof_auc": [macro_auc(yb[:, k:k + 1], fused_oof[:, k:k + 1], Ypool[done, k:k + 1] != 0.5) for k in range(12)]}, index=LABELS).round(4)
    json.dump(summary, open(f"{CFG.OUT_DIR}/metrics.json", "w"), indent=1, default=float)
    json.dump(cfg_dump, open(f"{CFG.OUT_DIR}/config.json", "w"), indent=1, default=float)
    print(json.dumps(summary, indent=1, default=float)); display(per)

## 7. Test inference → `submission.csv`

Every finished fold of every arm predicts the test studies (with mirror TTA); folds are averaged per arm, arms are
fused with the OOF weights. Studies with no readable series get the training prevalence. The output is checked
against `sample_submission.csv`.

In [ ]:
test = pd.read_csv(f"{DATA}/test.csv")
wdir = CFG.OUT_DIR if CFG.MODE == "train" else CFG.WEIGHTS_DIR
saved = json.load(open(f"{wdir}/config.json"))
if CFG.MODE == "infer":
    for k in ("SLOTS", "SPAN", "CROP_MM", "IMG", "PCT", "ALLOW_SLOT_REUSE", "D_MODEL", "TF_DEPTH", "TOTAL"):
        if k in saved: setattr(CFG, k, [tuple(s) for s in saved[k]] if k == "SLOTS" else saved[k])
    SLOT_OF = np.concatenate([[s] * n for s, (_, _, n) in enumerate(CFG.SLOTS)]); SLOT_START = np.cumsum([0] + [n for _, _, n in CFG.SLOTS])[:-1]
    SAG_SLOTS = [s for s, (p, _, _) in enumerate(CFG.SLOTS) if p == "Sagittal"]; N_SLOT = len(CFG.SLOTS); WIN_IDX, WIN_POS = window_table()
    CFG.AMP = CFG.AMP and DEVICE.type == "cuda"
prev = np.array(saved["prevalence"])
TVOL, TMASK = build_cache("test", test.StudyInstanceUID.tolist())
VOL, MASK = TVOL, TMASK
parts = {}
for arm, info in saved["arms"].items():
    ps = []
    for fn, h in zip(info["files"], info["sha256"]):
        path = f"{wdir}/{fn}"; assert sha256(path) == h, f"checkpoint changed: {fn}"
        model = make_model(info["backbone"], False).to(DEVICE).to(memory_format=torch.channels_last)
        model.load_state_dict(torch.load(path, map_location=DEVICE)); ps.append(fill_nan(predict(model, loader(np.arange(len(test)))), prev))
        del model; torch.cuda.empty_cache()
    parts[arm] = np.mean(ps, 0); print(f"arm {arm} ({info['backbone']}): {len(ps)} fold model(s)")
Wt = {a: np.array(w) for a, w in saved["fusion_weights"].items()}
if len(parts) > 1: P = ensemble(parts, Wt, "rank")             # weighted mean of per-column ranks, in (0, 1)
else: P = next(iter(parts.values()))
if CFG.OUTPUT == "rank": P = rankpct(P)
sub = pd.DataFrame(P, columns=LABELS); sub.insert(0, "StudyInstanceUID", test.StudyInstanceUID.values)
ss = pd.read_csv(f"{DATA}/sample_submission.csv"); sub = sub[ss.columns]
assert np.isfinite(sub[LABELS].values).all(), "non-finite predictions"
assert set(sub.StudyInstanceUID) == set(ss.StudyInstanceUID), "rows differ from sample_submission.csv"
sub.to_csv("submission.csv", index=False)
print(f"submission.csv: {sub.shape} | total {elapsed()}")
sub.head()